# 2. Create the AI Search index
Create a claims-evidence index: one row per evidence chunk, one contextual content vector, and explicit source/citation metadata. This notebook does **not** extract data, generate embeddings or upload records. Use the root `.venv` and `.env`. Use a new index name for this schema; an existing incompatible index is never replaced.

In [1]:
import os, json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from IPython.display import display
from azure.identity import DefaultAzureCredential
from azure.core.exceptions import ResourceNotFoundError
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchField, SearchFieldDataType, SearchIndex, VectorSearch, HnswAlgorithmConfiguration, HnswParameters, VectorSearchProfile, SemanticConfiguration, SemanticSearch, SemanticPrioritizedFields, SemanticField

ENV_FILE = Path(find_dotenv(usecwd=True, raise_error_if_not_found=True))
load_dotenv(ENV_FILE, override=True)
AZURE_SEARCH_SERVICE_ENDPOINT = os.getenv('AZURE_SEARCH_SERVICE_ENDPOINT', '').strip()
if not AZURE_SEARCH_SERVICE_ENDPOINT:
    raise ValueError(f'Set AZURE_SEARCH_SERVICE_ENDPOINT to the connected Search service endpoint in {ENV_FILE}; no index operation was attempted.')
AZURE_SEARCH_ADMIN_CREDENTIAL = DefaultAzureCredential()
index_name = os.getenv('AZURE_SEARCH_INDEX_NAME', 'claims-evidence-v1').lower().replace('_', '-')
azure_openai_model_dimensions = int(os.getenv('AZURE_OPENAI_EMBEDDING_DIMENSIONS', '3072'))
pd.set_option('display.max_rows', 5)
pd.set_option('display.max_columns', 6)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_seq_items', 3)
print('Index:', index_name)
print('Vector dimensions:', azure_openai_model_dimensions)

Index: claims-evidence-v1
Vector dimensions: 3072


## Step 1: Define and display the fields
These are exactly notebook 1's evidence columns. Claim/document identity supports scoping and grouping; source type and evidence kind support filtering. Local source paths, original aliases, parent emails and native figure regions support citations. Only title/header/content/filename are full-text searchable. Full vectors are hidden from search responses.

In [2]:
print('-' * 72)
print('Step 1: Index fields')
fields = [
    SearchField(name='id', type=SearchFieldDataType.String, key=True, filterable=True),
    SearchField(name='claim_id', type=SearchFieldDataType.String, filterable=True),
    SearchField(name='document_hash', type=SearchFieldDataType.String, filterable=True),
    SearchField(name='chunk_id', type=SearchFieldDataType.Int32, sortable=True),
    SearchField(name='title', type=SearchFieldDataType.String, searchable=True),
    SearchField(name='header', type=SearchFieldDataType.String, searchable=True),
    SearchField(name='content', type=SearchFieldDataType.String, searchable=True),
    SearchField(name='source_type', type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchField(name='content_kind', type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchField(name='filename', type=SearchFieldDataType.String, searchable=True, filterable=True),
    SearchField(name='source_path', type=SearchFieldDataType.String),
    SearchField(name='source_files', type=SearchFieldDataType.Collection(SearchFieldDataType.String)),
    SearchField(name='parent_emails', type=SearchFieldDataType.Collection(SearchFieldDataType.String), filterable=True),
    SearchField(name='figure_id', type=SearchFieldDataType.String, filterable=True),
    SearchField(name='source_region', type=SearchFieldDataType.String),
    SearchField(name='analyzer_id', type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchField(name='analyzer_api_version', type=SearchFieldDataType.String, filterable=True),
    SearchField(name='content_vector', type=SearchFieldDataType.Collection(SearchFieldDataType.Single), searchable=True, hidden=True, vector_search_dimensions=azure_openai_model_dimensions, vector_search_profile_name='claims-content-profile'),
]
for field in fields:
    for attribute in ['key', 'hidden', 'searchable', 'filterable', 'facetable', 'sortable']:
        setattr(field, attribute, bool(getattr(field, attribute)))
display(pd.DataFrame([{'name': field.name, 'type': field.type, 'filterable': bool(field.filterable), 'searchable': bool(field.searchable), 'dimensions': field.vector_search_dimensions} for field in fields]).head(5))
print('Fields:', len(fields), '|', ', '.join(field.name for field in fields))

------------------------------------------------------------------------
Step 1: Index fields


,name,type,filterable,searchable,dimensions
0,id,Edm.String,True,False,NaN
1,claim_id,Edm.String,True,False,NaN
2,document_hash,Edm.String,True,False,NaN
3,chunk_id,Edm.Int32,False,False,NaN
4,title,Edm.String,False,True,NaN


Fields: 18 | id, claim_id, document_hash, chunk_id, title, header, content, source_type, content_kind, filename, source_path, source_files, parent_emails, figure_id, source_region, analyzer_id, analyzer_api_version, content_vector


## Step 2: Configure vector and semantic search
Notebook 1 embeds title, section, evidence kind and content together. The searchable text fields, HNSW/cosine content vector and `claims-semantic` over title/content/header support semantic hybrid search in notebook 3: keyword text + query vector + semantic reranking with a claim filter. A claim filter scopes retrieval; it is not authorization.

Semantic ranker must also be enabled on the Search service in a supported region/tier. This index configuration does not enable or change the service feature. Notebook 3 requests 50 vector candidates and returns five reranked evidence results; it raises semantic errors rather than falling back to ordinary hybrid search.

In [3]:
print('-' * 72)
print('Step 2: Vector and semantic configuration')
vector_search = VectorSearch(algorithms=[HnswAlgorithmConfiguration(name='claims-hnsw', parameters=HnswParameters(metric='cosine'))], profiles=[VectorSearchProfile(name='claims-content-profile', algorithm_configuration_name='claims-hnsw')])
semantic_config = SemanticConfiguration(name='claims-semantic', prioritized_fields=SemanticPrioritizedFields(title_field=SemanticField(field_name='title'), content_fields=[SemanticField(field_name='content'), SemanticField(field_name='header')]))
semantic_search = SemanticSearch(configurations=[semantic_config], default_configuration_name='claims-semantic')
index = SearchIndex(name=index_name, fields=fields, vector_search=vector_search, semantic_search=semantic_search)

def ValidateIndexSchema(actual_index):
    actual_fields = {field.name: field for field in actual_index.fields}
    differences = []
    if set(actual_fields) != {field.name for field in fields}:
        differences.append('field names')
    for expected in fields:
        if expected.name not in actual_fields:
            continue
        actual = actual_fields[expected.name]
        for attribute in ['type', 'vector_search_dimensions', 'vector_search_profile_name']:
            if getattr(actual, attribute) != getattr(expected, attribute):
                differences.append(f'{expected.name}.{attribute}')
        for attribute in ['key', 'hidden', 'searchable', 'filterable', 'facetable', 'sortable']:
            if bool(getattr(actual, attribute)) != bool(getattr(expected, attribute)):
                differences.append(f'{expected.name}.{attribute}')
    profiles = actual_index.vector_search.profiles if actual_index.vector_search else []
    algorithms = actual_index.vector_search.algorithms if actual_index.vector_search else []
    if not any(profile.name == 'claims-content-profile' and profile.algorithm_configuration_name == 'claims-hnsw' for profile in profiles or []):
        differences.append('vector profile')
    if not any(algorithm.name == 'claims-hnsw' and algorithm.kind == 'hnsw' and algorithm.parameters and algorithm.parameters.metric == 'cosine' for algorithm in algorithms or []):
        differences.append('HNSW/cosine algorithm')
    configurations = actual_index.semantic_search.configurations if actual_index.semantic_search else []
    if not actual_index.semantic_search or actual_index.semantic_search.default_configuration_name != 'claims-semantic':
        differences.append('default semantic configuration')
    semantic = next((configuration for configuration in configurations or [] if configuration.name == 'claims-semantic'), None)
    prioritized = semantic.prioritized_fields if semantic else None
    if not prioritized or not prioritized.title_field or prioritized.title_field.field_name != 'title' or [field.field_name for field in prioritized.content_fields or []] != ['content', 'header']:
        differences.append('semantic configuration')
    if differences:
        raise ValueError('Incompatible claims-evidence index (' + ', '.join(differences) + '). Set AZURE_SEARCH_INDEX_NAME to a new name; the existing index was not changed.')

ValidateIndexSchema(index)
print('Vector: one contextual content vector, HNSW/cosine | Semantic: claims-semantic')

------------------------------------------------------------------------
Step 2: Vector and semantic configuration
Vector: one contextual content vector, HNSW/cosine | Semantic: claims-semantic


## Step 3: Create the index
This is the explicit Azure write. If the index already exists, its schema must match; it is not silently replaced.

In [4]:
print('-' * 72)
print('Step 3: Creating or checking the index')
index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=AZURE_SEARCH_ADMIN_CREDENTIAL)
try:
    existing = index_client.get_index(index_name)
except ResourceNotFoundError:
    index_client.create_index(index)
    print(index_name, 'created')
else:
    ValidateIndexSchema(existing)
    print(index_name, 'already exists with a compatible claims-evidence schema')
print('Next: run 03_populate_index.ipynb.')
index_client.close()
AZURE_SEARCH_ADMIN_CREDENTIAL.close()

------------------------------------------------------------------------
Step 3: Creating or checking the index
claims-evidence-v1 created
Next: run 03_populate_index.ipynb.
